In [1]:
%pip install -Uq langchain langchain-community langchain-openai langchain-text-splitters langchain-chroma chromadb pypdf python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.2/331.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.

In [ ]:
# =========================
# 0) 安裝套件（建議）
# =========================
# 在 Jupyter / terminal 執行：
# pip install -U langchain langchain-community langchain-openai langchain-text-splitters langchain-chroma chromadb pypdf python-dotenv


# =========================
# 1) 基本設定：API Key
# =========================
import os
from dotenv import load_dotenv

load_dotenv()  # 從 .env 讀取 OPENAI_API_KEY（建議不要在程式裡硬寫 key）
# .env 範例：
# OPENAI_API_KEY=sk-xxxxx


# =========================
# 2) 載入文件
# =========================
from langchain_community.document_loaders import TextLoader  # PyPDFLoader 也在這個包內
# from langchain_community.document_loaders import PyPDFLoader

# 這裡沿用你原本教學的「建立示例文件」
sample_text = """
公司遠端工作政策

為了提供員工更大的工作靈活性，本公司制定以下遠端工作政策：

1. 遠端工作資格
- 所有全職員工在入職滿 6 個月後，可申請遠端工作
- 部分職位可能因工作性質不適用遠端工作

2. 工作時間要求
- 遠端工作者須維持標準工作時間：上午 9 點至下午 6 點
- 必須在團隊核心會議時間（上午 10 點至下午 4 點）保持在線

3. 設備與技術要求
- 公司提供筆記型電腦和必要軟體
- 員工需確保穩定的網路連線
- 須安裝公司指定的安全軟體

4. 績效評估
- 遠端工作員工的績效評估標準與辦公室員工相同
- 採用目標導向的評估方式
""".strip()

with open("company_policy.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

loader = TextLoader("company_policy.txt", encoding="utf-8")
documents = loader.load()


# =========================
# 3) 文件分割（新版建議用 langchain-text-splitters）
# =========================
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
splits = text_splitter.split_documents(documents)


# =========================
# 4) 建立向量資料庫（Chroma）
# =========================
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings()  # 使用 OPENAI_API_KEY

vector_store = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db",
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)


# =========================
# 5) 建立「檢索 + 回答」的 RAG Chain（官方推薦組裝方式）
# =========================
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

llm = ChatOpenAI(
    model="gpt-4o-mini",  # 你也可以換成你帳號可用的模型
    temperature=0.1,
)

system_prompt = (
    "你是一個專業的企業政策諮詢助理。"
    "請只根據提供的文件內容回答問題。"
    "如果文件中沒有相關資訊，請明確回答「文件中沒有提到」。"
    "\n\n"
    "文件內容：\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)


# =========================
# 6) 執行查詢
# =========================
query = "公司的遠端工作政策是什麼？"
result = rag_chain.invoke({"input": query})

print("問題：", query)
print("\n答案：")
print(result["answer"])

print("\n=== 來源片段（前 200 字）===")
for i, doc in enumerate(result["context"], start=1):
    print(f"\n來源 {i}:")
    print(doc.page_content[:200])